# Tutorial 7-2: Decision Tree -- Visualizing Splitting Criteria
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
### Why Gini and Entropy are Better than Misclassification Error

## 1. The Search for "Purity"
When a Decision Tree splits a region $R_p$ into two children $R_1$ and $R_2$, it wants to maximize the decrease in "impurity." In lecture, we discussed three ways to measure this impurity $L$:

1. **Misclassification Error:** $1 - max(p, 1-p)$
2. **Gini Index:** $2p(1-p)$
3. **Cross-Entropy:** $-p \log_2(p) - (1-p) \log_2(1-p)$

**Objective:** In this tutorial, we will:
1. Define these functions mathematically.
2. Plot them across all possible values of $p \in [0, 1]$.
3. Understand why the **strict concavity** of Gini and Entropy makes them superior for tree growth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def misclassification_error(p):
    return 1 - np.max([p, 1 - p], axis=0)

def gini_index(p):
    return 2 * p * (1 - p)

def cross_entropy(p):
    # Avoid log(0) by adding a tiny epsilon
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

# Create a range of probabilities from 0 to 1
p = np.linspace(0, 1, 500)

error_vals = [misclassification_error(i) for i in p]
gini_vals = gini_index(p)
entropy_vals = cross_entropy(p)

## 2. Plotting the Impurity Curves
Let's recreate the "Impurity Functions" graph from the lecture slides.

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(p, error_vals, label='Misclassification Error', linestyle='--')
plt.plot(p, gini_vals, label='Gini Index', color='green')
plt.plot(p, entropy_vals, label='Cross-Entropy', color='orange')

plt.title("Impurity Measures for Binary Classification")
plt.xlabel("Probability $p$ of Class 1")
plt.ylabel("Impurity Value $L$")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Why Concavity Matters
Notice the shape of the curves:
* **Misclassification Error** is composed of two straight lines. It is not "strictly concave."
* **Gini and Entropy** are smooth curves that bulge significantly higher in the middle.

### The Split Benefit
Imagine a parent node with a 50/50 mix of classes ($p=0.5$). If we split it into two nodes—one with $p=0.8$ and one with $p=0.2$—the average impurity of the children is lower than the parent.

Because Gini and Entropy are strictly concave, any split that results in different class distributions in the children **guarantees** a decrease in the weighted average loss. Misclassification error, being linear, might show no improvement even if the split is helpful for creating purer nodes later in the tree.

## 4. Exercise: Calculate the Gain
Using the provided functions, calculate the Gini Gain for the following scenario from the slides:
* **Parent:** 400 positive, 100 negative
* **Child 1:** 300 positive, 100 negative
* **Child 2:** 100 positive, 0 negative

In [ ]:
def calculate_gain(p_parent, p_c1, p_c2, n_total, n_c1, n_c2):
    l_parent = gini_index(p_parent)
    l_children = (n_c1/n_total)*gini_index(p_c1) + (n_c2/n_total)*gini_index(p_c2)
    return l_parent - l_children

gain = calculate_gain(p_parent=0.8, p_c1=0.75, p_c2=1.0, n_total=500, n_c1=400, n_c2=100)
print(f"Gini Information Gain for the split: {gain:.4f}")